# AI401 — Experiment 3: Forward Chaining Inference Engine
## Data-Driven Reasoning / ZOOKEEPER-style Animal Identification

**Roll No.: U23AI114**

This notebook implements the forward chaining inference engine described in the laboratory handout:
- Working memory / fact set
- IF–THEN production rules
- Conflict-set construction
- First-applicable rule firing according to rule order
- Firing-sequence recording
- Animal identification
- Tiger, giraffe and zebra extensions
- Most-specific-class reporting
- Comparison with firing **all** applicable rules per cycle


## Aim

To implement a forward chaining inference engine and use it to identify an animal from observed features, in the manner of the classic ZOOKEEPER system.

Forward chaining is data-driven: it starts from known facts and repeatedly applies applicable rules to derive new facts until a goal is reached or no rule can fire.

In [ ]:
from dataclasses import dataclass
from typing import List, Set, Tuple

@dataclass(frozen=True)
class Rule:
    identifier: str
    conditions: frozenset
    conclusion: str

    def applicable(self, facts: Set[str]) -> bool:
        return self.conditions.issubset(facts) and self.conclusion not in facts


class ForwardChainingEngine:
    def __init__(self, rules: List[Rule]):
        self.rules = rules

    def conflict_set(self, facts: Set[str]) -> List[Rule]:
        return [r for r in self.rules if r.applicable(facts)]

    def run_first(self, initial_facts: Set[str], goals: Set[str] = None,
                  verbose=True):
        facts = set(initial_facts)
        firing_sequence = []
        trace = []

        while True:
            conflicts = self.conflict_set(facts)

            if verbose:
                print("WM:", sorted(facts))
                print("Conflict set:", [r.identifier for r in conflicts])

            if goals and facts.intersection(goals):
                break

            if not conflicts:
                break

            rule = conflicts[0]
            facts.add(rule.conclusion)
            firing_sequence.append(rule.identifier)
            trace.append((rule.identifier, rule.conditions, rule.conclusion))

            if verbose:
                print(f"FIRE {rule.identifier}: "
                      f"{' AND '.join(sorted(rule.conditions))} -> {rule.conclusion}")
                print()

        return facts, firing_sequence, trace

    def run_all(self, initial_facts: Set[str], goals: Set[str] = None,
                verbose=True):
        facts = set(initial_facts)
        firing_sequence = []
        cycle_trace = []

        while True:
            conflicts = self.conflict_set(facts)

            if verbose:
                print("WM:", sorted(facts))
                print("Conflict set:", [r.identifier for r in conflicts])

            if goals and facts.intersection(goals):
                break

            if not conflicts:
                break

            fired_this_cycle = []
            # Fire all rules applicable at the beginning of this cycle.
            for rule in conflicts:
                if rule.conclusion not in facts:
                    facts.add(rule.conclusion)
                    firing_sequence.append(rule.identifier)
                    fired_this_cycle.append(rule.identifier)

            cycle_trace.append(fired_this_cycle)

            if verbose:
                print("FIRE ALL:", fired_this_cycle)
                print()

            if not fired_this_cycle:
                break

        return facts, firing_sequence, cycle_trace


## 1. Animal-identification knowledge base

The rules below use the classic hierarchical style needed for forward chaining. The initial demonstration is designed so that the important chain is:

**R1 → R4 → R6**

which derives intermediate facts such as **is mammal** and **is carnivore**, followed by a specific animal identification.

In [ ]:
rules = [
    Rule("R1", frozenset({"has hair"}), "is mammal"),
    Rule("R2", frozenset({"gives milk"}), "is mammal"),
    Rule("R3", frozenset({"is mammal", "has hoofs"}), "is ungulate"),
    Rule("R4", frozenset({"is mammal", "eats meat"}), "is carnivore"),
    Rule("R5", frozenset({"is carnivore", "has tawny color"}), "is big cat"),
    Rule("R6", frozenset({"is big cat", "has black stripes"}), "tiger"),
    Rule("R7", frozenset({"is ungulate", "has long neck", "has long legs"}), "giraffe"),
    Rule("R8", frozenset({"has black stripes", "is not carnivore"}), "zebra"),
]

engine = ForwardChainingEngine(rules)
print("Knowledge base:")
for r in rules:
    print(f"{r.identifier}: IF {' AND '.join(sorted(r.conditions))} THEN {r.conclusion}")


## 2. Main demonstration

Initial observed features are chosen to identify a tiger through the chained intermediate conclusions.

The engine:
1. Initializes working memory.
2. Builds the conflict set.
3. Fires the first applicable rule.
4. Adds its conclusion to working memory.
5. Repeats until the animal is identified or no rule can fire.

In [ ]:
initial_facts = {
    "has hair",
    "eats meat",
    "has tawny color",
    "has black stripes"
}

print("=" * 70)
print("FORWARD CHAINING DEMONSTRATION")
print("=" * 70)

final_facts, firing_sequence, trace = engine.run_first(
    initial_facts,
    goals={"tiger", "giraffe", "zebra"},
    verbose=True
)

print("FINAL WORKING MEMORY:")
print(sorted(final_facts))
print("\nFIRING SEQUENCE:")
print(" -> ".join(firing_sequence))
print("\nIDENTIFICATION:")
animals = {"tiger", "giraffe", "zebra"}
identified = sorted(final_facts.intersection(animals))
print(identified[0] if identified else "No identification can be established.")


## 3. Exercise 1 — Add rules and facts for tiger, giraffe and zebra

The knowledge base contains rules for all three requested animals:
- **Tiger:** hair + eats meat → mammal/carnivore → big cat; stripes → tiger
- **Giraffe:** mammal + hoofs + long neck + long legs → giraffe
- **Zebra:** black stripes + not carnivore → zebra

The following tests verify each identification separately.

In [ ]:
test_cases = {
    "Tiger": {
        "has hair", "eats meat", "has tawny color", "has black stripes"
    },
    "Giraffe": {
        "has hair", "has hoofs", "has long neck", "has long legs"
    },
    "Zebra": {
        "has black stripes", "is not carnivore"
    }
}

exercise1_results = []

for expected, facts in test_cases.items():
    final, seq, _ = engine.run_first(
        facts, goals={"tiger", "giraffe", "zebra"}, verbose=False
    )
    animals_found = sorted(final.intersection({"tiger", "giraffe", "zebra"}))
    identified = animals_found[0] if animals_found else "No identification"

    exercise1_results.append({
        "Expected animal": expected,
        "Identified animal": identified,
        "Firing sequence": " -> ".join(seq),
        "Correct": identified.lower() == expected.lower()
    })

import pandas as pd
display(pd.DataFrame(exercise1_results))


## 4. Exercise 2 — No direct animal identification

When the supplied facts do not satisfy an animal rule, the engine should stop when the conflict set becomes empty.

Instead of falsely identifying an animal, it reports the **most specific class established** by the available facts.

In [ ]:
def most_specific_class(facts):
    # Priority from most specific useful intermediate class to general class.
    priority = [
        "is big cat",
        "is carnivore",
        "is ungulate",
        "is mammal"
    ]
    for cls in priority:
        if cls in facts:
            return cls
    return "No class established"

unknown_facts = {
    "has hair",
    "eats meat",
    "has tawny color"
    # No stripes, so tiger cannot be established.
}

final_unknown, seq_unknown, _ = engine.run_first(
    unknown_facts,
    goals={"tiger", "giraffe", "zebra"},
    verbose=True
)

animals_found = sorted(final_unknown.intersection({"tiger", "giraffe", "zebra"}))

print("FINAL WORKING MEMORY:", sorted(final_unknown))
print("FIRING SEQUENCE:", " -> ".join(seq_unknown) if seq_unknown else "None")

if animals_found:
    print("IDENTIFIED ANIMAL:", animals_found[0])
else:
    print("IDENTIFIED ANIMAL: No identification can be established.")
    print("MOST SPECIFIC CLASS:", most_specific_class(final_unknown))


## 5. Exercise 3 — Fire ALL applicable rules per cycle

The original algorithm fires the **first** rule from the conflict set. This modified engine fires every rule that is applicable at the beginning of each cycle.

The two approaches may produce different firing sequences because several rules can be applicable simultaneously. The final logical consequences can still be equivalent.

In [ ]:
print("=" * 70)
print("FIRST-APPLICABLE VERSION")
print("=" * 70)

first_facts, first_sequence, _ = engine.run_first(
    initial_facts,
    goals={"tiger", "giraffe", "zebra"},
    verbose=True
)

print("Sequence:", first_sequence)

print("\n" + "=" * 70)
print("FIRE-ALL VERSION")
print("=" * 70)

all_facts, all_sequence, all_cycles = engine.run_all(
    initial_facts,
    goals={"tiger", "giraffe", "zebra"},
    verbose=True
)

print("Sequence:", all_sequence)
print("Cycles:", all_cycles)


In [ ]:
comparison = pd.DataFrame([
    {
        "Method": "First applicable",
        "Firing sequence": " -> ".join(first_sequence),
        "Final animal": next(
            (x for x in ["tiger", "giraffe", "zebra"] if x in first_facts),
            "None"
        ),
        "Number of rule firings": len(first_sequence)
    },
    {
        "Method": "All applicable per cycle",
        "Firing sequence": " -> ".join(all_sequence),
        "Final animal": next(
            (x for x in ["tiger", "giraffe", "zebra"] if x in all_facts),
            "None"
        ),
        "Number of rule firings": len(all_sequence)
    }
])

display(comparison)


# Final Observations

1. The forward chaining engine starts from observed features in working memory and progressively derives new facts.
2. The conflict set changes whenever a new conclusion is added to working memory.
3. The recorded firing sequence provides an explanation trace of how the final conclusion was reached.
4. Intermediate facts such as **is mammal**, **is carnivore**, **is big cat**, and **is ungulate** demonstrate hierarchical reasoning.
5. If no rule can fire before an animal is identified, the engine correctly reports that no identification can be established.
6. Firing all applicable rules in a cycle can shorten the number of cycles and can change the order of the recorded firing sequence, while preserving the conclusions supported by the rules.

# Conclusion

The experiment successfully implements a forward chaining inference engine using IF–THEN production rules. The system performs data-driven reasoning by repeatedly matching rules against the growing working memory, resolving the conflict set according to rule order, firing rules, and recording the inference chain. The extended knowledge base identifies tiger, giraffe, and zebra, while unknown cases are handled without forcing an unsupported animal identification. The all-rules-per-cycle modification further demonstrates how the inference strategy affects the firing sequence.

In [ ]:
# Compact lab-record summary
summary = pd.DataFrame([{
    "Experiment": "Forward Chaining Inference Engine",
    "Roll No.": "U23AI114",
    "Reasoning type": "Data-driven / Forward chaining",
    "Animals covered": "Tiger, Giraffe, Zebra",
    "Conflict resolution": "First applicable rule",
    "Extension": "Fire all applicable rules per cycle"
}])
display(summary)
